In [1]:
import os, json, pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
TEMP_DATA_STORAGE = "../../temp_data/aiops22"          # 统一根目录
RAW_DATA_ROOT     = f"{TEMP_DATA_STORAGE}/raw_data"   # 你 extract_log_features 输出在这里

# analysis/log 下：你已经生成过 patterns / istio_words / miners（这里会继续写 TF-IDF/统计结果）
ANALYSIS_LOG_DIR  = f"{TEMP_DATA_STORAGE}/analysis/log"

# dataset/log 下：这段脚本会输出 z-score 结果和 windowed dataset
DATASET_LOG_DIR   = f"{TEMP_DATA_STORAGE}/dataset/log"

os.makedirs(ANALYSIS_LOG_DIR, exist_ok=True)
os.makedirs(DATASET_LOG_DIR, exist_ok=True)

print("RAW_DATA_ROOT    =", RAW_DATA_ROOT)
print("ANALYSIS_LOG_DIR =", ANALYSIS_LOG_DIR)
print("DATASET_LOG_DIR  =", DATASET_LOG_DIR)

RAW_DATA_ROOT    = ../../temp_data/aiops22/raw_data
ANALYSIS_LOG_DIR = ../../temp_data/aiops22/analysis/log
DATASET_LOG_DIR  = ../../temp_data/aiops22/dataset/log


In [3]:
service_order_list = [
    "adservice","cartservice","checkoutservice","currencyservice","emailservice",
    "frontend","paymentservice","productcatalogservice","recommendationservice","shippingservice"
]

pod_list = [
    'adservice-0', 
    'adservice-1', 
    'adservice-2', 
    'adservice2-0', 
    'cartservice-0', 
    'cartservice-1', 
    'cartservice-2', 
    'cartservice2-0', 
    'checkoutservice-0', 
    'checkoutservice-1', 
    'checkoutservice-2', 
    'checkoutservice2-0', 
    'currencyservice-0', 
    'currencyservice-1', 
    'currencyservice-2', 
    'currencyservice2-0', 
    'emailservice-0', 
    'emailservice-1', 
    'emailservice-2', 
    'emailservice2-0', 
    'frontend-0', 
    'frontend-1', 
    'frontend-2', 
    'frontend2-0', 
    'paymentservice-0', 
    'paymentservice-1', 
    'paymentservice-2', 
    'paymentservice2-0', 
    'productcatalogservice-0', 
    'productcatalogservice-1', 
    'productcatalogservice-2', 
    'productcatalogservice2-0', 
    'recommendationservice-0', 
    'recommendationservice-1', 
    'recommendationservice-2', 
    'recommendationservice2-0', 
    'shippingservice-0', 
    'shippingservice-1', 
    'shippingservice-2', 
    'shippingservice2-0'
]

node_list = ["node-1","node-2","node-3","node-4","node-5","node-6"]

service_language_dict = {
    "adservice": "Java",
    "cartservice": "C#",
    "checkoutservice": "Go",
    "currencyservice": "Node.js",
    "emailservice": "Python",
    "frontend": "Go",
    "paymentservice": "Node.js",
    "productcatalogservice": "Go",
    "recommendationservice": "Python",
    "shippingservice": "Go",
}

def rename_pod2service(pod_name: str) -> str:
    return pod_name.replace("2-0", "").replace("-0", "").replace("-1", "").replace("-2", "")

def rename_service2pod(service: str):
    return [f"{service}-0", f"{service}-1", f"{service}-2", f"{service}2-0"]

# language -> service list（源代码用的就是这个结构）
language_service = {}
for svc, lang in service_language_dict.items():
    language_service.setdefault(lang, []).append(svc)

languages = sorted(language_service.keys())
languages

['C#', 'Go', 'Java', 'Node.js', 'Python']

In [4]:
def list_cases(raw_data_root: str):
    """
    返回 [(dataset_type, date, cloudbed, raw_log_dir), ...]
    """
    cases = []
    if not os.path.isdir(raw_data_root):
        raise FileNotFoundError(raw_data_root)

    for dataset_type in sorted(os.listdir(raw_data_root)):
        dt_path = os.path.join(raw_data_root, dataset_type)
        if not os.path.isdir(dt_path):
            continue
        for date in sorted(os.listdir(dt_path)):
            date_path = os.path.join(dt_path, date)
            if not os.path.isdir(date_path):
                continue
            for cloudbed in sorted(os.listdir(date_path)):
                cb_path = os.path.join(date_path, cloudbed)
                raw_log_dir = os.path.join(cb_path, "raw_log")
                if os.path.isdir(raw_log_dir):
                    cases.append((dataset_type, date, cloudbed, raw_log_dir))
    return cases

cases = list_cases(RAW_DATA_ROOT)
cases[:5], len(cases)

([('test_data',
   '2022-05-01',
   'cloudbed',
   '../../temp_data/aiops22/raw_data/test_data/2022-05-01/cloudbed/raw_log'),
  ('test_data',
   '2022-05-03',
   'cloudbed',
   '../../temp_data/aiops22/raw_data/test_data/2022-05-03/cloudbed/raw_log'),
  ('test_data',
   '2022-05-05',
   'cloudbed',
   '../../temp_data/aiops22/raw_data/test_data/2022-05-05/cloudbed/raw_log'),
  ('test_data',
   '2022-05-07',
   'cloudbed',
   '../../temp_data/aiops22/raw_data/test_data/2022-05-07/cloudbed/raw_log'),
  ('test_data',
   '2022-05-09',
   'cloudbed',
   '../../temp_data/aiops22/raw_data/test_data/2022-05-09/cloudbed/raw_log')],
 15)

In [5]:
def load_container_log_features(raw_log_dir: str) -> pd.DataFrame:
    f = os.path.join(raw_log_dir, "log_patterns_count.csv")
    if not os.path.exists(f):
        raise FileNotFoundError(f)
    return pd.read_csv(f)

def load_istio_log_features(raw_log_dir: str) -> pd.DataFrame:
    f = os.path.join(raw_log_dir, "istio_word_count.csv")
    if not os.path.exists(f):
        raise FileNotFoundError(f)
    return pd.read_csv(f)

# quick sanity
dt, date, cb, raw_log_dir = cases[0]
df_c = load_container_log_features(raw_log_dir)
df_i = load_istio_log_features(raw_log_dir)
df_c.shape, df_i.shape, df_c.columns[:5], df_i.columns[:5]

((1440, 1133),
 (1440, 6890),
 Index(['timestamp', 'adservice-0; <log pattern 0>',
        'adservice-0; <log pattern 1>', 'adservice-0; <log pattern 2>',
        'adservice-0; <log pattern 3>'],
       dtype='object'),
 Index(['timestamp', 'total', '"Prometheus/2.12.0"', '"Python/THttpClient',
        '"adservice.ts:8088"'],
       dtype='object'))

In [6]:
def extract_entity_feature_name(feature_name: str) -> str:
    """
    源代码：把 'pod; xxx' 统一成 'service; xxx' 的 exact_feature_name（用于统计量 key）
    """
    cmdb_id = feature_name.split(';')[0].replace('2-0', '').replace('-0', '').replace('-1', '').replace('-2', '')
    return f'{cmdb_id};{feature_name.split(";")[1]}'

In [7]:
'''
    输入：
    1.log_patterns.json
    log_pattern_dict={
        "Java":    [template_1, template_2, ...],
        "Go":      [...],
        "Python":  [...],
        ...
    }
    ==================================================================================================
    ==================================================================================================
    
    输出：
    1.container_log_tf_idf_statistic.json
    result_dict[lang]={
        "analysis": {"idf": {}, "max_tf": {}, "max_tf_idf": {}},
        "raw_tf_idf": {"tf": {}, "tf_idf": {}},
    }
    result_dict[lang][analysis][idf][i]=模板i在lang所对应的所有pod上的idf得分
   
    2.container_log_selected_patterns.json
    selected_pattern_dict[lang]=List
    表示在lang所对应的所有pod上，max_tf_idf得分超过threshold的模板编号列表
    根据TF-IDF得分，筛选在每种编程语言上，重要的日志模板
'''
def calculate_log_pattern_tf_idf(
    cases,
    skip_bad_case=True,
    bad_case_date="2022-03-24",
    bad_case_cloudbeds=("cloudbed-1", "cloudbed-2"),
    threshold=0.3,
):
    # load patterns
    patterns_json = os.path.join(ANALYSIS_LOG_DIR, "log_patterns.json")
    with open(patterns_json, "r", encoding="utf-8") as f:
        log_pattern_dict = json.load(f)
    '''
        log_pattern_dict={
            "Java":    [template_1, template_2, ...],
            "Go":      [...],
            "Python":  [...],
            ...
        }
    '''
    result_dict = {}
    temp_dict = {}
    selected_pattern_dict = {}

    '''
        container_log_tf_idf_statistic.json
        result_dict[lang]={
            "analysis": {"idf": {}, "max_tf": {}, "max_tf_idf": {}},
            "raw_tf_idf": {"tf": {}, "tf_idf": {}},
        }
        result_dict[lang][analysis][idf][i]=模板i在lang所对应的所有pod上的idf得分
        ==================================================================================================
        temp_dict[lang][i]=occurrence_list
        表示第i个模板，在lang所对应的所有pod上，每分钟出现的次数。len(occurrence_list) = len(timestamp_list)
        ==================================================================================================
        container_log_selected_patterns.json
        selected_pattern_dict[lang]=List
        表示在lang所对应的所有pod上，max_tf_idf得分超过threshold的模板编号列表
    '''

    # 为每个语言的每个日志模板准备“计分容器”和“中间数据”
    for lang, pat_list in log_pattern_dict.items():
        result_dict[lang] = {
            "analysis": {"idf": {}, "max_tf": {}, "max_tf_idf": {}},
            "raw_tf_idf": {"tf": {}, "tf_idf": {}},
        }
        temp_dict[lang] = [[] for _ in range(len(pat_list))]
        selected_pattern_dict[lang] = []

    # iterate all train-like cases (source code skips test)
    for dataset_type, date, cloudbed, raw_log_dir in tqdm(cases, desc="scan cases"):
        if dataset_type == "test":
            continue
        if skip_bad_case and date == bad_case_date and cloudbed in bad_case_cloudbeds:
            continue

        feature_df = load_container_log_features(raw_log_dir)
        '''
            log_patterns_count.csv
            最后生成的横表表头为
            timestamp {pod}; <log pattern {i} {pod}; <log pattern {i} ...
            记录每分钟，每个pod对应template发生的次数
        '''
        # 记录每种编程语言上，每个日志模板在每分钟出现的总次数
        # temp_dict[lang][i]=List[T]
        for lang, svc_list in language_service.items():
            pat_list = log_pattern_dict.get(lang, [])
            if len(pat_list) == 0:
                continue

            pods = []
            for svc in svc_list:
                pods.extend(rename_service2pod(svc))

            for i in range(len(pat_list)):
                cols = [f"{pod}; <log pattern {i}>" for pod in pods]
                cols = [c for c in cols if c in feature_df.columns]
                # 筛选feature_df统计存在的{pod}; <log pattern {i}
                if len(cols) == 0:
                    occurrence_list = [0] * feature_df.shape[0]
                else:                    
                    occurrence_list = feature_df.loc[:, cols].apply(lambda x: x.sum(), axis=1).tolist()
                    # 在某一个language对应的所有pod上，模板i在每一分钟发生的总次数
                temp_dict[lang][i].extend(occurrence_list)



    '''
        temp_dict[lang][i]=occurrence_list
        表示第i个模板，在lang所对应的所有pod上，每分钟出现的次数。len(occurrence_list) = len(timestamp_list)
        temp_dict[lang]=num_templates * len_timestamp
    '''
    # compute tf-idf stats & select
    # 在每一种编程语言内部，根据 TF-IDF 得分筛选“重要”的日志模板
    for lang in temp_dict.keys():
        sum_list = np.sum(temp_dict[lang], axis=0)
        # len(sum_list) = len(timestamp_list), 在lang所对应的所有pod上，每分钟所出现的总模板数
        for i in range(len(temp_dict[lang])):
            arr = np.array(temp_dict[lang][i], dtype=float)
            # 表示第i个模板，在lang所对应的所有pod上，每分钟出现的次数 T
            if not np.any(sum_list):
                tf = arr
            else:
                tf = np.true_divide(arr, sum_list)
                # 在lang所对应的所有pod上，每分钟模板i占所有模板的出现比例，len(tf) = T

            idf = np.log(arr.shape[0] / (1 + np.count_nonzero(arr)))
            # idf = log( 总时间点数 / (1 + 出现过的时间点数) )

            # 和源代码一致：max_tf 取 top30 mean
            tf_valid = tf[~np.isnan(tf)]
            if tf_valid.size == 0:
                max_tf = 0.0
            else:
                max_tf = np.sort(tf_valid)[-30:].mean()
                # 在所有分钟里，模板 i 的 TF 值最大的 30 个分钟的平均

            max_tf_idf = max_tf * idf
            # 模板i在lang所对应的所有pod上的重要性得分

            result_dict[lang]["analysis"]["idf"][str(i)] = float(idf)
            result_dict[lang]["analysis"]["max_tf"][str(i)] = float(max_tf)
            result_dict[lang]["analysis"]["max_tf_idf"][str(i)] = float(max_tf_idf)

            if max_tf_idf > threshold:
                selected_pattern_dict[lang].append(i)

    with open(os.path.join(ANALYSIS_LOG_DIR, "container_log_tf_idf_statistic.json"), "w", encoding="utf-8") as f:
        json.dump(result_dict, f, indent=2)

    with open(os.path.join(ANALYSIS_LOG_DIR, "container_log_selected_patterns.json"), "w", encoding="utf-8") as f:
        json.dump(selected_pattern_dict, f, indent=2)

    return selected_pattern_dict

selected_patterns = calculate_log_pattern_tf_idf(cases)
selected_patterns

scan cases: 100%|██████████| 15/15 [00:25<00:00,  1.70s/it]
/tmp/ipykernel_3398587/1927610453.py:128: RuntimeWarning: invalid value encountered in divide
  tf = np.true_divide(arr, sum_list)


{'C#': [32, 36],
 'Go': [16, 21, 22],
 'Java': [3],
 'Node.js': [],
 'Python': [14, 16]}

In [8]:
'''
    输出：
    1.container_log_statistic.json
    statistic_dict[{svc}; <log pattern {i}>]={
        "mean": float(mean),
        "std": float(std),
        "percentile_1": float(percentile_1),
        "q1": float(q1),
        "median": float(median),
        "q3": float(q3),
        "percentile_99": float(percentile_99),
        "valid_ratio": valid_ratio,
    }
    统计在每个微服务svc上被选择得分超过阈值的日志模板相关信息
'''
def calculate_container_log_pattern_statistic(
    cases,
    skip_bad_case=True,
    bad_case_date="2022-03-24",
    bad_case_cloudbeds=("cloudbed-1", "cloudbed-2"),
):
    with open(os.path.join(ANALYSIS_LOG_DIR, "container_log_selected_patterns.json"), "r", encoding="utf-8") as f:
        selected_log_pattern_dict = json.load(f)

    # max_tf_idf得分超过threshold的{svc}; <log pattern {i}>
    selected_log_pattern_features = []
    for lang, svc_list in language_service.items():
        for svc in svc_list:
            for i in selected_log_pattern_dict.get(lang, []):
                selected_log_pattern_features.append(f"{svc}; <log pattern {i}>")

    statistic_dict = {}
    data_dict = {}

    '''
        data_dict[{svc}; <log pattern {i}>]=List len=T
        在该svc上每分钟发生的模板i的数量
    '''
    for dataset_type, date, cloudbed, raw_log_dir in tqdm(cases, desc="stat container log"):
        if dataset_type == "test":
            continue
        if skip_bad_case and date == bad_case_date and cloudbed in bad_case_cloudbeds:
            continue

        feature_df = load_container_log_features(raw_log_dir)
        '''
            log_patterns_count.csv
            最后生成的横表表头为
            timestamp {pod}; <log pattern {i}> {pod}; <log pattern {i}> ...
            记录每分钟，每个pod对应template发生的次数
        '''
        # 聚合成service的，统计在不同service上，不同模板每分钟发生的频率
        # 如果一个service对应不同pod上都有某个模板，它不会累加起来，而是聚合成一个长列表，因为后面要统计mean、std
        for feature_name in feature_df.columns:
            if feature_name == "timestamp":
                continue
            exact_feature_name = extract_entity_feature_name(feature_name)
            # {svc}; <log pattern {i}>
            if exact_feature_name in selected_log_pattern_features:
                data_dict.setdefault(exact_feature_name, [])
                data_dict[exact_feature_name].extend(feature_df[feature_name].tolist())

    # 计算不同service上出现过日志模板的统计量
    for feature_name, log_data in data_dict.items():
        log_data = np.array(log_data, dtype=float)
        median = np.nanmedian(log_data)
        percentile_1 = np.nanpercentile(log_data, 1)
        percentile_99 = np.nanpercentile(log_data, 99)
        q1 = np.nanpercentile(log_data, 25)
        q3 = np.nanpercentile(log_data, 75)
        mean = np.nanmean(log_data)
        std = np.nanstd(log_data)
        valid_ratio = float(np.count_nonzero(~np.isnan(log_data)) / len(log_data))

        statistic_dict[feature_name] = {
            "mean": float(mean),
            "std": float(std),
            "percentile_1": float(percentile_1),
            "q1": float(q1),
            "median": float(median),
            "q3": float(q3),
            "percentile_99": float(percentile_99),
            "valid_ratio": valid_ratio,
        }

    with open(os.path.join(ANALYSIS_LOG_DIR, "container_log_statistic.json"), "w", encoding="utf-8") as f:
        json.dump(statistic_dict, f, indent=2)

    return statistic_dict

container_log_stat = calculate_container_log_pattern_statistic(cases)
len(container_log_stat)

stat container log: 100%|██████████| 15/15 [00:00<00:00, 18.17it/s]


19

In [9]:
'''
    输入：
    1.container_log_selected_patterns.json
    selected_pattern_dict[lang]=List
    表示在lang所对应的所有pod上，max_tf_idf得分超过threshold的模板编号列表
    2.container_log_statistic.json
    statistic_dict[{svc}; <log pattern {i}>]={
        "mean": float(mean),
        "std": float(std),
        "percentile_1": float(percentile_1),
        "q1": float(q1),
        "median": float(median),
        "q3": float(q3),
        "percentile_99": float(percentile_99),
        "valid_ratio": valid_ratio,
    }
    统计在每个微服务svc上被选择得分超过阈值的日志模板相关信息
    ==================================================================================================
    ==================================================================================================
    
    输出：
    1.z_scored_container_log_features.pkl
    out[dataset_type][date][cloudbed]["container_log_features"] = pd.DataFrame(selected_df)
    pd.DataFrame(selected_df)表头为，列=T，值为(raw_series - mean) / std
    timestamp {pod}; {lang}; <log pattern {i}>     {pod}; {lang}; <log pattern {i}>...
'''
# 每个时间窗口 × 每个 pod × 每个被选中的模板
# 计算Z-归一化值，并非该lang选中的模板设为0
# {pod}; {lang}; <log pattern {i}>，这里同时出现lang、pod是因为日志模板i是属于lang的第i个日志模板
# XL∈Rt×nl
def z_score_container_log_data(
    cases,
    skip_bad_case=True,
    bad_case_date="2022-03-24",
    bad_case_cloudbeds=("cloudbed-1", "cloudbed-2"),
):
    with open(os.path.join(ANALYSIS_LOG_DIR, "container_log_selected_patterns.json"), "r", encoding="utf-8") as f:
        selected_log_pattern_feature_dict = json.load(f)

    with open(os.path.join(ANALYSIS_LOG_DIR, "container_log_statistic.json"), "r", encoding="utf-8") as f:
        statistic_dict = json.load(f)

    # output dict: out[dataset_type][date][cloudbed]["container_log_features"]=DataFrame
    out = {}

    for dataset_type, date, cloudbed, raw_log_dir in tqdm(cases, desc="zscore container log"):
        if skip_bad_case and date == bad_case_date and cloudbed in bad_case_cloudbeds:
            continue

        feature_df = load_container_log_features(raw_log_dir)

        out.setdefault(dataset_type, {}).setdefault(date, {})[cloudbed] = {"container_log_features": None}
        selected_df = {"timestamp": feature_df["timestamp"]}

        for pod in pod_list:
            svc = rename_pod2service(pod)
            svc_lang = service_language_dict[svc]

            for lang, selected_pattern_list in selected_log_pattern_feature_dict.items():
                for i in selected_pattern_list:
                    if lang == svc_lang:
                        exact_feature_name = extract_entity_feature_name(f"{pod}; <log pattern {i}>")
                        raw_series = feature_df.get(f"{pod}; <log pattern {i}>", pd.Series([0]*len(feature_df)))
                        std = statistic_dict[exact_feature_name]["std"]
                        mean = statistic_dict[exact_feature_name]["mean"]
                        if std != 0:
                            selected_df[f"{pod}; {lang}; <log pattern {i}>"] = (raw_series - mean) / std
                        else:
                            selected_df[f"{pod}; {lang}; <log pattern {i}>"] = [0 for _ in range(feature_df.shape[0])]
                    else:
                        selected_df[f"{pod}; {lang}; <log pattern {i}>"] = [0 for _ in range(feature_df.shape[0])]

        out[dataset_type][date][cloudbed]["container_log_features"] = pd.DataFrame(selected_df)


    with open(os.path.join(DATASET_LOG_DIR, "z_scored_container_log_features.pkl"), "wb") as f:
        pickle.dump(out, f)

    return out

z_container = z_score_container_log_data(cases)
print("saved:", os.path.join(DATASET_LOG_DIR, "z_scored_container_log_features.pkl"))

zscore container log: 100%|██████████| 15/15 [00:01<00:00,  7.55it/s]


saved: ../../temp_data/aiops22/dataset/log/z_scored_container_log_features.pkl


In [10]:
# 筛选得分大于阈值的候选词是吧
def calculate_word_tf_idf(
    cases,
    skip_bad_case=True,
    bad_case_date="2022-03-24",
    bad_case_cloudbeds=("cloudbed-1", "cloudbed-2"),
    threshold=0.04,
):
    with open(os.path.join(ANALYSIS_LOG_DIR, "istio_words.json"), "r", encoding="utf-8") as f:
        raw_istio_words = json.load(f)["word_set"]

    istio_words = [w for w in raw_istio_words if ("latency=" not in w and "ttl=" not in w)]

    result_dict = {
        "analysis": {"idf": {}, "max_tf": {}, "max_tf_idf": {}},
        "raw_tf_idf": {"tf": {}, "tf_idf": {}},
    }
    temp_dict = {w: [] for w in istio_words}

    for dataset_type, date, cloudbed, raw_log_dir in tqdm(cases, desc="scan istio tfidf"):
        if dataset_type == "test":
            continue
        if skip_bad_case and date == bad_case_date and cloudbed in bad_case_cloudbeds:
            continue

        feature_df = load_istio_log_features(raw_log_dir)

        # 源代码：temp_dict[word].extend((feature_df[word] / feature_df['total']).tolist())
        for w in istio_words:
            if w not in feature_df.columns:
                temp_dict[w].extend([0.0] * feature_df.shape[0])
            else:
                denom = feature_df["total"].replace(0, np.nan)
                temp_dict[w].extend((feature_df[w] / denom).fillna(0.0).tolist())

    selected_word_list = []
    for w in istio_words:
        temp = np.array(temp_dict[w], dtype=float)
        idf = np.log(temp.shape[0] / (1 + np.count_nonzero(temp)))

        max_tf = np.sort(temp)[-30:].mean() if temp.size else 0.0
        max_tf_idf = max_tf * idf

        result_dict["analysis"]["idf"][w] = float(idf)
        result_dict["analysis"]["max_tf"][w] = float(max_tf)
        result_dict["analysis"]["max_tf_idf"][w] = float(max_tf_idf)

        result_dict["raw_tf_idf"]["tf"][w] = temp.tolist()
        result_dict["raw_tf_idf"]["tf_idf"][w] = (temp * idf).tolist()

        if max_tf_idf > threshold:
            selected_word_list.append(w)

    with open(os.path.join(ANALYSIS_LOG_DIR, "istio_log_tf_idf_statistic.json"), "w", encoding="utf-8") as f:
        json.dump(result_dict, f, indent=2)

    with open(os.path.join(ANALYSIS_LOG_DIR, "istio_selected_words.json"), "w", encoding="utf-8") as f:
        json.dump(selected_word_list, f, indent=2)

    return selected_word_list

selected_words = calculate_word_tf_idf(cases)
len(selected_words), selected_words[:10]

scan istio tfidf: 100%|██████████| 15/15 [00:07<00:00,  1.99it/s]


(17,
 ['"transport:',
  '=',
  'DC',
  'Error',
  'UH',
  'connection',
  'desc',
  'dial',
  'dialing',
  'downstream_remote_disconnect'])

In [11]:
'''
    输出：
    1.z_scored_istio_features.pkl
    out[dataset_type][date][cloudbed]["istio_log_features"] = pd.DataFrame(selected_df)
    pd.DataFrame(selected_df)表头为，列=T，值为(raw_series - mean) / std
    timestamp {pod}; {w}     {pod}; {w}...

    关键词和模板处理逻辑基本一致
'''
# 每一分钟、每个 pod、每个被选中的 istio 关键词做 z-score 标准化
def calculate_istio_log_statistic(
    cases,
    skip_bad_case=True,
    bad_case_date="2022-03-24",
    bad_case_cloudbeds=("cloudbed-1", "cloudbed-2"),
):
    with open(os.path.join(ANALYSIS_LOG_DIR, "istio_selected_words.json"), "r", encoding="utf-8") as f:
        selected_words = json.load(f)

    statistic_dict = {}
    data_dict = {}

    for dataset_type, date, cloudbed, raw_log_dir in tqdm(cases, desc="stat istio log"):
        if dataset_type == "test":
            continue
        if skip_bad_case and date == bad_case_date and cloudbed in bad_case_cloudbeds:
            continue

        feature_df = load_istio_log_features(raw_log_dir)

        for pod in pod_list:
            for w in selected_words:
                col = f"{pod}; {w}"
                if col not in feature_df.columns:
                    continue
                exact_feature_name = extract_entity_feature_name(f"{pod}; {w}")
                data_dict.setdefault(exact_feature_name, [])
                data_dict[exact_feature_name].extend(feature_df[col].tolist())

    for feature_name, log_data in data_dict.items():
        log_data = np.array(log_data, dtype=float)
        median = np.nanmedian(log_data)
        percentile_1 = np.nanpercentile(log_data, 1)
        percentile_99 = np.nanpercentile(log_data, 99)
        q1 = np.nanpercentile(log_data, 25)
        q3 = np.nanpercentile(log_data, 75)
        mean = np.nanmean(log_data)
        std = np.nanstd(log_data)
        valid_ratio = float(np.count_nonzero(~np.isnan(log_data)) / len(log_data))

        statistic_dict[feature_name] = {
            "mean": float(mean),
            "std": float(std),
            "percentile_1": float(percentile_1),
            "q1": float(q1),
            "median": float(median),
            "q3": float(q3),
            "percentile_99": float(percentile_99),
            "valid_ratio": valid_ratio,
        }

    with open(os.path.join(ANALYSIS_LOG_DIR, "istio_log_statistic.json"), "w", encoding="utf-8") as f:
        json.dump(statistic_dict, f, indent=2)

    return statistic_dict

def z_score_istio_log_data(
    cases,
    skip_bad_case=True,
    bad_case_date="2022-03-24",
    bad_case_cloudbeds=("cloudbed-1", "cloudbed-2"),
):
    with open(os.path.join(ANALYSIS_LOG_DIR, "istio_selected_words.json"), "r", encoding="utf-8") as f:
        selected_words = json.load(f)

    with open(os.path.join(ANALYSIS_LOG_DIR, "istio_log_statistic.json"), "r", encoding="utf-8") as f:
        statistic_dict = json.load(f)

    out = {}

    for dataset_type, date, cloudbed, raw_log_dir in tqdm(cases, desc="zscore istio log"):
        if skip_bad_case and date == bad_case_date and cloudbed in bad_case_cloudbeds:
            continue

        feature_df = load_istio_log_features(raw_log_dir)

        out.setdefault(dataset_type, {}).setdefault(date, {})[cloudbed] = {"istio_log_features": None}
        selected_df = {"timestamp": feature_df["timestamp"]}

        for pod in pod_list:
            for w in selected_words:
                col = f"{pod}; {w}"
                exact_feature_name = extract_entity_feature_name(f"{pod}; {w}")
                if col not in feature_df.columns or exact_feature_name not in statistic_dict:
                    selected_df[col] = [0 for _ in range(feature_df.shape[0])]
                    continue
                raw_series = feature_df[col]
                mean = statistic_dict[exact_feature_name]["mean"]
                std  = statistic_dict[exact_feature_name]["std"]
                if std != 0:
                    selected_df[col] = (raw_series - mean) / std
                else:
                    selected_df[col] = [0 for _ in range(feature_df.shape[0])]

        out[dataset_type][date][cloudbed]["istio_log_features"] = pd.DataFrame(selected_df)
      
    
    with open(os.path.join(DATASET_LOG_DIR, "z_scored_istio_features.pkl"), "wb") as f:
        pickle.dump(out, f)

    return out

_ = calculate_istio_log_statistic(cases)
z_istio = z_score_istio_log_data(cases)
print("saved:", os.path.join(DATASET_LOG_DIR, "z_scored_istio_features.pkl"))

zscore istio log: 100%|██████████| 15/15 [00:08<00:00,  1.80it/s]

saved: ../../temp_data/aiops22/dataset/log/z_scored_istio_features.pkl


In [5]:
# ====== 你 HolisticRCA 工程代码路径（照你之前 sys.path.append 用法） ======
PROJECT_CODE_PATH = "/workspace/project/working/2023/HolisticRCA/code"
import sys
if PROJECT_CODE_PATH not in sys.path:
    sys.path.append(PROJECT_CODE_PATH)

# window_size_list 来自源代码的 self.window_size_list，你在 notebook 里直接定义即可
window_size_list = [5, 10, 15]  # 你按自己的实验配置填

# 尝试导入 TimeIntervalLabelGenerator（模式A）
try:
    from data_filter.CCF_AIOps_challenge_2022.service.time_interval_label_generator import TimeIntervalLabelGenerator
    HAS_TIME_INTERVAL_GENERATOR = True
except Exception as e:
    HAS_TIME_INTERVAL_GENERATOR = False
    print("[WARN] TimeIntervalLabelGenerator not available, fallback to sliding windows.")

[WARN] TimeIntervalLabelGenerator not available, fallback to sliding windows.


In [6]:
window_size_list = [5, 10, 15]
def build_case_index_for_generator(cases):
    """
    为 fallback 生成器准备：按 (date, cloudbed) 归类 case
    """
    case_map = {}
    for dataset_type, date, cloudbed, raw_log_dir in cases:
        case_map.setdefault(dataset_type, {}).setdefault(date, {})[cloudbed] = raw_log_dir
    return case_map

case_map = build_case_index_for_generator(cases)

def fallback_time_intervals(window_size_min: int, dataset_type: str):
    """
    fallback：对每个 case 生成整天滑窗 intervals
    interval tuple: (date, cloudbed, st, et) 其中 st/et 是 timestamp(秒)
    """
    out = []
    for date, cb_dict in case_map.get(dataset_type, {}).items():
        for cloudbed, raw_log_dir in cb_dict.items():
            # 用 container_log_features 的 timestamp 作为真值
            df = load_container_log_features(raw_log_dir)
            ts = df["timestamp"].values
            # ts 是每分钟一个点，窗口长度 = window_size_min
            # st/et 采用 timestamp 值（秒）
            for i in range(0, len(ts) - window_size_min):
                st = int(ts[i])
                et = int(ts[i + window_size_min])  # 半开区间 [st, et)
                out.append((date, cloudbed, st, et))
    return out

# 全天候有问题
# def get_time_interval_list(window_size, data_type):
#     """
#     data_type: 'train_valid' or 'test'
#     """
#     if HAS_TIME_INTERVAL_GENERATOR:
#         return TimeIntervalLabelGenerator().get_time_interval_label(window_size)["time_interval"][data_type]
#     else:
#         # fallback：把 train_valid 映射到 train，test 映射到 test
#         dataset_type = "training_data_with_faults" if data_type == "train_valid" else "test_data"
#         return fallback_time_intervals(window_size, dataset_type)

TIME_INTERVAL_PKL_DIR = os.path.join(TEMP_DATA_STORAGE, "dataset", "time_interval_and_label")

def get_time_interval_list(window_size, data_type):
    """
    data_type: 'train_valid' or 'test'
    intervals item format: [date, cloudbed, st, et]
    """
    fp = os.path.join(TIME_INTERVAL_PKL_DIR, f"time_interval_window_size_{window_size}.pkl")
    with open(fp, "rb") as f:
        obj = pickle.load(f)
    # 这里返回的就是 list[list]，每个元素 [date, cloudbed, st, et]
    return obj["time_interval"][data_type]

def get_time_interval_log_data(st, et, df):
    # 源代码：np.array(df.query(f'{st} <= timestamp < {et}').iloc[:, df.columns != "timestamp"].values)
    return np.array(df.query(f"{st} <= timestamp < {et}").iloc[:, df.columns != "timestamp"].values)

def generate_log_data():
    # load z-scored dicts
    with open(os.path.join(DATASET_LOG_DIR, "z_scored_istio_features.pkl"), "rb") as f:
        istio_z = pickle.load(f)
    '''
        out[dataset_type][date][cloudbed]["istio_log_features"] = pd.DataFrame(selected_df)
        pd.DataFrame(selected_df)表头为，列=T，值为(raw_series - mean) / std
        timestamp {pod}; {w}     {pod}; {w}...
    '''
    with open(os.path.join(DATASET_LOG_DIR, "z_scored_container_log_features.pkl"), "rb") as f:
        container_z = pickle.load(f)
    '''
        out[dataset_type][date][cloudbed]["container_log_features"] = pd.DataFrame(selected_df)
        pd.DataFrame(selected_df)表头为，列=T，值为(raw_series - mean) / std
        timestamp {pod}; {lang}; <log pattern {i}>     {pod}; {lang}; <log pattern {i}>...
    '''
    for window_size in tqdm(window_size_list, desc="window_size"):
        log_dict = {}

        # entity_features & log_names（按源代码结构）
        entity_features = []
        log_name_list = []
        record_features = True

        for node in node_list:
            entity_features.append((node, (0, 0)))
        for svc in service_order_list:
            entity_features.append((svc, (0, 0)))

        for data_type in ["train_valid", "test"]:
            time_interval_label_list = get_time_interval_list(window_size, data_type)
            # 5、10、15分钟的时间列表
            # [('2022-03-20', 'cloudbed-1', 1647705600, 1647705900)]
            
            log_dict[data_type] = []
            for time_interval in tqdm(time_interval_label_list, desc=f"{data_type} intervals", leave=False):
                # 源代码 time_interval: (date, cloudbed, st, et)
                date, cloudbed, st, et = time_interval[0], time_interval[1], int(time_interval[2]), int(time_interval[3])

                # 对应 dataset_type：train_valid -> train，test -> test
                dataset_type = "training_data_with_faults" if data_type == "train_valid" else "test_data"

                container_df = container_z[dataset_type][date][cloudbed]["container_log_features"]
                '''
                    pd.DataFrame(selected_df)表头为，列=T，值为(raw_series - mean) / std
                    timestamp {pod}; {lang}; <log pattern {i}>     {pod}; {lang}; <log pattern {i}>...
                '''
                istio_df     = istio_z[dataset_type][date][cloudbed]["istio_log_features"]
                '''
                    pd.DataFrame(selected_df)表头为，列=T，值为(raw_series - mean) / std
                    timestamp {pod}; {w}     {pod}; {w}...
                '''
                # 筛选五分钟、十分钟、十五分钟内的数据
                # 在同一个时间窗口内，把日志模板特征和关键词特征拼接在一块
                container_temp = get_time_interval_log_data(st, et, container_df)
                istio_temp     = get_time_interval_log_data(st, et, istio_df)
                temp = np.concatenate((container_temp, istio_temp), axis=1)

                # 拼接container_df和istio_df的表头
                # {pod}; {lang}; <log pattern {i}> ... {pod}; {w} ...
                temp_name_list = list(container_df.columns[container_df.columns != "timestamp"])
                temp_name_list.extend(list(istio_df.columns[istio_df.columns != "timestamp"]))

                data = []
                log_feature_name_list = []
                feature_index = 0

                for svc in service_order_list:
                    pods = rename_service2pod(svc)

                    for pod in pods:
                        # 为所有pod设置统一的特征顺序
                        if len(log_feature_name_list) == 0:
                            # 收集pod相关的模板或关键词列表，格式为; {lang}; <log pattern {i}>或; {w}
                            log_feature_name_list = [
                                fname.split(pod)[1] for fname in temp_name_list if pod in fname
                            ]
                        pod_related_log_name_list = [f"{pod}{suffix}" for suffix in log_feature_name_list]


                        '''
                            data = [
                              pod1-feature1,
                              pod1-feature2,
                              ...
                              podN-featureM
                            ]
                            按这个顺序抽取特征，形状为T
                        '''
                        for fname in pod_related_log_name_list:
                            data.append(temp[:, temp_name_list.index(fname)])
                        # 当下pod在每一分钟，该log_feature的z-score值
                        '''
                            data = [
                              array(T_window),  # pod1-feature1
                              array(T_window),  # pod1-feature2
                              ...
                              array(T_window),  # podN-featureM
                            ]
                        '''

                        '''
                            记录哪些列属于哪些实体
                            adservice-0 → (0, 15)
                            adservice-1 → (15, 30)
                        '''
                        if record_features:
                            entity_features.append((pod, (feature_index, feature_index + len(pod_related_log_name_list))))
                            # 这个 pod 的 log 特征位于 [start_col, end_col) 这个区间，列索引
                            feature_index += len(pod_related_log_name_list)
                            log_name_list.extend(pod_related_log_name_list)
                            # {pod}; {lang}; <log pattern {i}> ... {pod}; {w} ...
 

                log_dict[data_type].append(np.array(data).transpose())
                record_features = False

        out_path = os.path.join(DATASET_LOG_DIR, f"log_window_size_{window_size}.pkl")

        '''
            log_data = {
                "train_valid": [array1, array2, ...],
                "test":        [array1, array2, ...]
            }
            每个array形状为shape = (T_window, F)，其中F = 总日志特征数
            ================================================================
            entity_features 每个实体对应的列区间
            [
                ("node-1", (0, 0)),
                ...
                ("adservice", (0, 0)),   # service占位
                ...
                ("adservice-0", (0, 15)),
                ("adservice-1", (15, 30)),
                ...
            ]
            =================================================================
            log_names 每一列的真实含义，长度为F
            [
              "adservice-0; Java; <log pattern 3>",
              "adservice-0; Java; <log pattern 7>",
              ...
              "adservice-0; error",
              ...
            ]

        '''
        with open(out_path, "wb") as f:
            pickle.dump({
                "log_data": log_dict,
                "entity_features": entity_features,
                "log_names": log_name_list,
            }, f)

        print("[OK] saved:", out_path)

generate_log_data()

train_valid intervals: 100%|██████████| 300/300 [00:26<00:00, 11.50it/s]
                                                                        
train_valid intervals:   1%|          | 2/300 [00:00<00:27, 11.03it/s]

[OK] saved: ../../temp_data/aiops22/dataset/log/log_window_size_5.pkl



train_valid intervals: 100%|██████████| 300/300 [00:26<00:00, 11.51it/s]
                                                                        
train_valid intervals:   0%|          | 1/300 [00:00<00:30,  9.84it/s]

[OK] saved: ../../temp_data/aiops22/dataset/log/log_window_size_10.pkl



train_valid intervals: 100%|█████████▉| 299/300 [00:26<00:00, 11.44it/s]
                                                                        
window_size: 100%|██████████| 3/3 [02:44<00:00, 54.86s/it]       

[OK] saved: ../../temp_data/aiops22/dataset/log/log_window_size_15.pkl
